# Lab 3 · Exercise A — On/off with a Gaussian constraint

Two ways to model the uncertainty on the background $b$:
an **auxiliary measurement** (Poisson, as in Lab 2) or a **constraint term** (Gaussian, from “$3.8 \pm 0.6$”).

We compute the significance of the top-quark excess ($n_{\rm on}=17$, $n_{\rm off}=40$, $\tau=10.53$) with both.

In [ ]:
import ROOT
%jsroot on

import numpy as np # for sqrt

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name


In [ ]:
non, noff, tau_val = 17, 40, 10.53

### A) Auxiliary measurements or control regions or on/off problem

Auxiliary measurements or control regions can be used to estimate or reduce the effect of systematic uncertainties

At least one parameter must be shared between signal and control regions.

The **signal region** and **control region** are *not fundamentally different*, they are just two different channels. If the model for the auxiliary measurement (or control region) were available, it could and should be included as an additional channel in a combined (or simultaneous) model. In case of number counting analysis (on/off probleam) just the product of PDF_on and PDF_off is enough.

(DONE) A simple **counting experiment** with an uncertain background ${\mathbf{ν}}_{\mathbf{B}}$ 
with **some control sample** for background estimation (**on/off problem**)

$\text{Pois}\left( n_{SR} \middle| \nu_{S} + \nu_{B} \right)$ The distribution of the number of events in the SIGNAL REGION

- $\nu_{S}$ : true, unknown signal rate (*parameter of interest*)
- $\nu_{B}$ : true, unknown background in the signal region (*nuisance parameter*)
- $n_{SR}$: number of events in the signal region (*observable*)

Then, we need some estimate for the background $\nu_{B}$ to make any useful inference about $\nu_{S}$.

*This may come from some control sample with no signal contamination and populated by the same background processes.*

$\text{Pois}\left( n_{CR} \middle| \tau\nu_{B} \right)$ The distribution of the number of events in the CONTROL REGION

- $n_{CR}$: number of events in the control region (observable)\
- $\tau$ : factor to extrapolate the background from the signal to the control region (constant)

the total probability model is the product:\
$f_{simult}\left( n_{SR},n_{CR} \right|\nu_{S},\ \nu_{B}) = \text{Pois}\left( n_{SR} \middle| \nu_{S} + \nu_{B} \right)\text{~Pois}\left( n_{CR} \middle| \tau\nu_{B} \right)$

In [ ]:
# Parameters of interest and nuisance parameters
s_p   = ROOT.RooRealVar("s_p", "signal", max(non - noff / tau_val, 0.5), 0, 200)
b_p   = ROOT.RooRealVar("b_p", "background", max(noff / tau_val, 0.5), 0.01, 200)
tau   = ROOT.RooRealVar("tau", "tau", tau_val)
tau.setConstant(True)

# Observables (both n_on and n_off are measured data)
n_on_p  = ROOT.RooRealVar("n_on_p", "n_on", 0, 300)
n_off   = ROOT.RooRealVar("n_off", "n_off", 0, 600)

# Model: Pois(n_on | s + b) * Pois(n_off | tau * b)
mu_on_p  = ROOT.RooFormulaVar("mu_on_p", "@0+@1", ROOT.RooArgList(s_p, b_p))
mu_off   = ROOT.RooFormulaVar("mu_off", "@0*@1", ROOT.RooArgList(tau, b_p))
p_on_p   = ROOT.RooPoisson("p_on_p", "p_on", n_on_p, mu_on_p)
p_off    = ROOT.RooPoisson("p_off", "p_off", n_off, mu_off)
pdf_pois = ROOT.RooProdPdf("pdf_pois", "pdf_pois", ROOT.RooArgList(p_on_p, p_off))


We generate a the observed data set. Since we have a counting model, the data set will contain only one event and the observables n_on_p, n_off will have our desired value .

In [ ]:
# Dataset and NLL
data_pois = ROOT.RooDataSet("data_pois", "data_pois", ROOT.RooArgSet(n_on_p, n_off))
n_on_p.setVal(non)
n_off.setVal(noff)
data_pois.add(ROOT.RooArgSet(n_on_p, n_off))

# NLL
nll_pois = pdf_pois.createNLL(data_pois)

### B) Constraint terms

In most cases involving background and systematic uncertainties, a detailed probability model for an auxiliary measurement is not available.

Often, we **only have an estimate** ${\mathbf{a}}_{\mathbf{p}}$ (e.g., a central value, or a guess) for a **parameter** ${\mathbf{α}}_{\mathbf{p}}$ along with an uncertainty, typically a $\pm 1\ \sigma$ variation.

TO DO: A simple **counting experiment** with an uncertain background ${\mathbf{ν}}_{\mathbf{B}}$
estimated as ${\mathbf{n}}_{\mathbf{B}} \pm {\mathbf{σ}}_{\mathbf{B}}$ from global fits, PDG booklet, ...
\
$\text{Pois}\left( n \middle| \nu_{S} + \nu_{B} \right)\$ The distribution of the number of events is a Poisson:

 - $\nu_{S}$ : true, unknown signal rate (*parameter of interest*)\
 - $\nu_{B}$ : true, unknown background (*nuisance parameter*)\
 - $n$ : number of events (*observable*)

$\nu_{B}$ is not not known exactly , but has uncertainty ${\mathbf{σ}}_{\mathbf{B}}$. Its estimated value is $n_{B}$

To express this uncertainty we add in the model a Gaussian constraint.\
We can interpret this term as having an additional measurement $n_{B}$ with an uncertainty ${\mathbf{σ}}_{\mathbf{B}}$: i.e. we have a likelihood for that measurement Gaus ($n_{B}$\| $\nu_{B}$, ${\mathbf{σ}}_{\mathbf{B}}$).

the total probability model is\
$f\left( n,n_{B} \right|\nu_{S},\ \nu_{B}) = \text{Pois}\left( n \middle| \nu_{S} + \nu_{B} \right)\text{~Gaus}\left( n_{B} \middle| \nu_{B},\ \sigma_{B} \right)$

**Note:** $\mathbf{n}$ **is an observable,**\
${\mathbf{n}}_{\mathbf{B}}$ **is a** ***global observable*** **(it originates from an external measurement)**

In [ ]:
b0_val  = noff / tau_val              # 3.80
sigma_b = np.sqrt(noff) / tau_val     # 0.60

# Parameters and global observable
s_g  = ROOT.RooRealVar("s_g", "signal", max(non - b0_val, 0.5), 0, 200)
b_g  = ROOT.RooRealVar("b_g", "background", b0_val, 0.01, 200)
b0   = ROOT.RooRealVar("b0", "global observable", b0_val, 0, 200)
b0.setConstant(True)
sb   = ROOT.RooRealVar("sb", "sigma_b", sigma_b)
sb.setConstant(True)

# Observable (only n_on is in the dataset; b0 is a global observable)
n_on_g = ROOT.RooRealVar("n_on_g", "n_on", 0, 300)

# TODO: build Pois(n_on | s + b) * Gauss(b0 | b, sigma_b)
mu_on_g   = None   # TODO: ROOT.RooFormulaVar  s_g + b_g
p_on_g    = None   # TODO: ROOT.RooPoisson(n_on_g, mu_on_g)
constr    = None   # TODO: ROOT.RooGaussian(b0, b_g, sb)
pdf_gauss = None   # TODO: ROOT.RooProdPdf of p_on_g and constr



We generate a the observed data set. Since we have a counting model, the data set will contain only one event and the observables n_on_p, n_off will have our desired value .

In [ ]:
# Dataset and NLL (note GlobalObservables!)
data_gauss = ROOT.RooDataSet("data_gauss", "data_gauss", ROOT.RooArgSet(n_on_g))
n_on_g.setVal(non)
data_gauss.add(ROOT.RooArgSet(n_on_g))

# NLL
nll_gauss = pdf_gauss.createNLL(data_gauss, ROOT.RooFit.GlobalObservables(ROOT.RooArgSet(b0)))

## A2 · The significance

To test the **background-only (null) hypothesis** ($H_0: s = 0$) against the **signal + background (alternative) hypothesis** ($H_1: s > 0$), we compute the profile likelihood test statistic $q_0$ and the asymptotic significance $Z_0$:

$$q_0 = 2 \left[ -\ln L(s=0, \hat{\hat{b}}) + \ln L(\hat{s}, \hat{b}) \right], \qquad Z_0 = \sqrt{q_0}$$

In RooFit (`nll` $= -\ln L$), we proceed in three steps:

1. **Signal + background fit (alternative hypothesis, unconditional):** ensure $s$ is floating (`s.setConstant(False)`), set clean starting values (`s = 10`, `b = 4`), and minimize `nll` $\rightarrow$ `nll_free` ($= -\ln L(\hat{s}, \hat{b})$).
2. **Background-only fit (null hypothesis, conditional):** fix $s = 0$ (`s.setVal(0); s.setConstant(True)`), reset the starting point for $b$ (`b = 4`), and re-minimize with respect to $b$ alone $\rightarrow$ `nll_null` ($= -\ln L(0, \hat{\hat{b}})$). Always release $s$ (`s.setConstant(False)`) afterward so later fits start from a clean state.
3. **Significance:** compute $Z_0 = \sqrt{2(\text{nll\_null} - \text{nll\_free})}$.

> **Why reset starting values (`s = 10, b = 4`)?** Numerical minimizers like Minuit2 need an initial guess to start searching. Resetting `s` and `b` before each fit guarantees reproducibility and prevents a previous fit (for example, one stuck on a boundary like $s = 0$) from biasing the next minimization.



The calculation for the **Poisson auxiliary model** is shown below; repeat the exact same steps for the **Gaussian constraint model**.

In [ ]:
# --- 1. Significance for Poisson auxiliary model ---

s_p.setConstant(False); s_p.setVal(10); b_p.setVal(4)
mn_p = ROOT.RooMinimizer(nll_pois)
mn_p.setPrintLevel(-1)
mn_p.minimize("Minuit2", "Migrad")
nll_pois_free = nll_pois.getVal()

s_p.setVal(0); s_p.setConstant(True); b_p.setVal(4)
mn_p.minimize("Minuit2", "Migrad")
nll_pois_null = nll_pois.getVal()
s_p.setConstant(False)

Zp = float(np.sqrt(max(2 * (nll_pois_null - nll_pois_free), 0)))


In [ ]:
# --- 2. Significance for Gaussian constraint model ---
# TODO: repeat the free fit (s_g, b_g floating) and null fit (s_g = 0 fixed) on nll_gauss
nll_gauss_free = None   # TODO
nll_gauss_null = None   # TODO

Zg = float(np.sqrt(max(2 * (nll_gauss_null - nll_gauss_free), 0)))

In [ ]:
print("Z0 auxiliary Poisson  : %.2f" % Zp)
print("Z0 Gaussian constraint: %.2f" % Zg)

check("z0 implemented", Zp is not None and Zg is not None)
check("Poisson-aux significance about 4.6", abs(Zp - 4.57) < 0.15)
check("Gaussian constraint agrees when n_off is large", abs(Zg - Zp) < 0.15)

## A3 · A poor control region
Same $\hat b=3.8$, but measured with only $n_{\rm off}=4$ events ($\tau=1.053$): the Gaussian constraint now has $\sigma_b=1.9$.
Which model gives the larger significance? Which one is right?

In [ ]:
# define new values
noff2, tau2 = 4, 1.053
b0_val2  = noff2 / tau2              # 3.80
sigma_b2 = np.sqrt(noff2) / tau2     # 1.90

# Update values in the Poisson model
n_off.setVal(noff2)
tau.setVal(tau2)
data_pois2 = ROOT.RooDataSet("data_pois2", "data_pois2", ROOT.RooArgSet(n_on_p, n_off))
data_pois2.add(ROOT.RooArgSet(n_on_p, n_off))
nll_pois2 = pdf_pois.createNLL(data_pois2)

# Update values in the Gaussian model
b0.setVal(b0_val2)
sb.setVal(sigma_b2)
nll_gauss2 = pdf_gauss.createNLL(data_gauss, ROOT.RooFit.GlobalObservables(ROOT.RooArgSet(b0)))

# TODO: compute Zp2 and Zg2 following the exact same steps as in A3
Zp2 = None   # TODO
Zg2 = None   # TODO

print("poor control region:  Poisson aux Z = %.2f   Gaussian Z = %.2f" % (Zp2, Zg2))


In [ ]:
check("the Poisson auxiliary measurement gives about 3.1", abs(Zp2 - 3.06) < 0.2)
check("the Gaussian constraint is too optimistic here", Zg2 > Zp2 + 0.2)

## Questions (2–3 lines each)
1. Why do the two models agree for $n_{\\rm off}=40$ and not for $n_{\\rm off}=4$?
2. In the Gaussian model, what is $b_0$ and why is it **not** in the dataset? What would change in a toy study?
3. You read “background $0.5\\pm 0.4$” in a paper and have no control data. Which constraint would you use, and why not a Gaussian?